# STI Pipeline Performance Dashboard
This dashboard aggregates metrics from all notebooks in the **Service Ticket Intelligence** project, providing a comparative analysis of model performance across iterations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setting Aesthetics
sns.set_theme(style="white", palette="viridis")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['axes.titlesize'] = 18


In [ ]:
# Load Registry
csv_path = 'notebook_evaluations.csv'
df = pd.read_csv(csv_path)

# Ensure numeric types
metrics = ['Silhouette', 'Davies_Bouldin', 'Calinski_Harabasz', 'Inertia']
df[metrics] = df[metrics].apply(pd.to_numeric, errors='coerce')

# Sort by Silhouette (Higher is better)
df_sorted = df.sort_values(by='Silhouette', ascending=True)

display(df)

## 1. Silhouette Score Comparison (Density)
**Goal**: Maximize the score. A higher Silhouette score indicates that objects are well matched to their own cluster and poorly matched to neighboring clusters.

In [ ]:
plt.figure(figsize=(12, 6))
colors = sns.color_palette("viridis", len(df_sorted))
ax = sns.barplot(data=df_sorted, x='Silhouette', y='Notebook_Name', palette=colors)

plt.title('Silhouette Score: TF-IDF vs Semantic Pipeline')
plt.xlabel('Silhouette Score (Higher is Better)')
plt.ylabel('Notebook')

# Annotate values
for p in ax.patches:
    ax.annotate(f"{p.get_width():.4f}", (p.get_width(), p.get_y()+0.5), padding=5, xytext=(5, 0), textcoords='offset points')

plt.xlim(0, max(df['Silhouette'])*1.2)
plt.grid(axis='x', alpha=0.3)
plt.show()

## 2. Davies-Bouldin Index (Separation)
**Goal**: Minimize the score. A lower Davies-Bouldin index indicates a better partition of the cluster space.

In [ ]:
# Sort by DB (Lower is better, so we sort descending to keep best at top of bar chart)
df_db = df.sort_values(by='Davies_Bouldin', ascending=False)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_db, x='Davies_Bouldin', y='Notebook_Name', palette='magma')

plt.title('Davies-Bouldin Index: Cluster Separation')
plt.xlabel('Davies-Bouldin Score (Lower is Better)')
plt.ylabel('Notebook')

for p in ax.patches:
    ax.annotate(f"{p.get_width():.4f}", (p.get_width(), p.get_y()+0.5), padding=5, xytext=(5, 0), textcoords='offset points')

plt.xlim(0, max(df['Davies_Bouldin'])*1.2)
plt.grid(axis='x', alpha=0.3)
plt.show()

## 3. Cumulative Semantic Gain Trajectory
Tracking the evolution of performance across the implementation sequence.

In [ ]:
# Using the original order of notebooks (001 -> 005)
df['Notebook_Num'] = df['Notebook_Name'].str.extract('(\\d+)').astype(int)
df_traj = df.sort_values(by='Notebook_Num')

plt.figure(figsize=(14, 6))
plt.plot(df_traj['Notebook_Name'], df_traj['Silhouette'], marker='o', linewidth=3, markersize=10, label='Silhouette (Higher is Good)')
plt.plot(df_traj['Notebook_Name'], 1/df_traj['Davies_Bouldin'], marker='s', linewidth=3, markersize=10, label='Inverse DB (Higher is Good)')

plt.title('Performance Trajectory: Scaling the Pipeline')
plt.ylabel('Normalized Score')
plt.xticks(rotation=15)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Executive Summary
| Model | Primary Advantage | Top Metric |
| :--- | :--- | :--- |
| **Baseline (001)** | Low compute, High interpretability | Speed |
| **Semantic (002/003)** | Dense features, Better clusters | Silhouette Gain |
| **Advanced (004)** | Approximate Matching, Scale | Sub-second Inference |
| **StackExchange (005)** | Multilingual Support | Cross-Lingual Semantic Gap |